# Earshot Lite - M4 on a Kaggle TPU

Runs the **same, unchanged** data-parallel code that was tested on 8 simulated CPU devices, on a real 8-core TPU.

**Notebook settings:** Accelerator = **TPU** (v5e-8 or v3-8), Internet = **on**.

1. Pin JAX/Flax to the versions the repo is tested with
2. Download Speech Commands and build the feature cache (~3 min)
3. Scaling benchmark: training throughput on 1/2/4/8 TPU cores (~5 min)
4. Full 60-epoch DS-CNN training, data-parallel over 8 cores
5. Zip the results for download (`/kaggle/working/m4_results.zip`)

## 0. Get the code
Clones the public repo.

In [ ]:
REPO_URL = "https://github.com/Soren-Caron/earshot-lite.git"
REPO_DIR = "/kaggle/working/earshot-lite"

import os, shutil, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(os.getcwd())

## 1. Pin versions (before JAX is imported anywhere in this kernel)

In [ ]:
!pip install -q "jax[tpu]==0.11.2" flax==0.12.10 optax==0.2.8 tqdm matplotlib scipy

In [ ]:
import jax
print(jax.__version__, jax.devices())
assert jax.devices()[0].platform == "tpu", "switch the notebook accelerator to TPU"

## 2. Data

In [ ]:
!python -m earshot.data.download
!python -m earshot.data.build_cache

## 3. Scaling benchmark
Real jitted train step (AdamW + augmentation + BatchNorm) on device-resident synthetic batches, 1/2/4/8 cores, weak and strong scaling.

In [ ]:
!python -m earshot.scaling --steps 50

In [ ]:
from IPython.display import Image
import glob
Image(sorted(glob.glob("docs/m4_scaling_tpu*.png"))[-1])

## 4. Full training, data-parallel over 8 cores
Global batch 1024 (128 per core), 60 epochs. On 8 simulated CPU devices, 30 epochs with this recipe reached 93.8% test, against 95.1% for the batch-128 recipe. That's the same number of epochs but 8x fewer optimizer steps, so the model is under-trained; epochs are cheap on TPU, so this doubles them. The learning rate is scaled up 2x from the batch-128 recipe (3e-3 -> 6e-3, a bit under sqrt(8) scaling) with a 2-epoch warmup. The same flags were validated on 8 simulated CPU devices.

In [ ]:
!python -m earshot.train --model dscnn --devices 8 --batch-size 1024 --lr 6e-3 --warmup-epochs 2 --epochs 60 --out runs/dscnn-tpu8

Single-core baseline with the same recipe, for an end-to-end epoch-time comparison (input pipeline included):

In [ ]:
!python -m earshot.train --model dscnn --devices 1 --batch-size 1024 --lr 6e-3 --warmup-epochs 2 --epochs 3 --out runs/dscnn-tpu1

## 5. Collect results

In [ ]:
import json, zipfile, pathlib
for run in ["runs/dscnn-tpu8", "runs/dscnn-tpu1"]:
    h = [json.loads(l) for l in open(f"{run}/history.jsonl")]
    print(run, "epoch_s (median):", sorted(e["epoch_s"] for e in h)[len(h)//2],
          "samples/s:", round(sorted(e["samples_per_s"] for e in h)[len(h)//2]))
print("test acc (8 cores):", json.load(open("runs/dscnn-tpu8/metrics.json"))["test_acc"])

with zipfile.ZipFile("/kaggle/working/m4_results.zip", "w") as z:
    for p in [*pathlib.Path("results").glob("m4_*"), *pathlib.Path("docs").glob("m4_*")]:
        z.write(p)
    for run in ["runs/dscnn-tpu8", "runs/dscnn-tpu1"]:
        for f in ["metrics.json", "history.jsonl", "config.json", "curves.png", "confusion.png"]:
            if pathlib.Path(run, f).exists():
                z.write(pathlib.Path(run, f))
print("download /kaggle/working/m4_results.zip from the Output panel")